# Aula 13A · APIs REST

Esta semana apresenta o [capítulo 13 do site](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/). A ideia central: **o script pergunta ao sistema
em vez de esperar alguém exportar um arquivo**. Uma API REST responde a uma URL com
um código de status e um corpo em JSON — e o JSON você já sabe ler desde a Aula 08.

**Ao fim das duas noites você consegue:**

1. fazer um `GET` com `requests`, conferir o código de status e ler o corpo JSON;
2. filtrar no servidor com parâmetros;
3. consultar vários recursos sem que a falha de um derrube os outros.

**Noite A — a ideia** (1h40): 📝 mini-teste · 📟 chamado · 1. requisição e resposta ·
2. uma lista de recursos · 3. parâmetros · 4. o código de status · 5. falha por item
· 6. o que mudou desde ontem · 🚪 antes de sair

O chamado se resolve na [noite
B](https://colab.research.google.com/github/lacouth/python_telecom-site/blob/main/notebooks/aula13b-apis-rest.ipynb),
no encontro seguinte, que é laboratório: nenhum assunto novo, os 🎯 que ficaram e a
lista começada em sala.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica e uma solução
  estão recolhidas: tente antes de abrir.

A célula ⚙️ desta aula também **põe no ar a API simulada da Maré Net**, dentro da
própria sessão, e guarda o endereço dela em `API`. Se o Colab reiniciar, rode-a de
novo.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")


# --- dados desta aula ---
# simulador: API REST da Maré Net — não precisa ler (é o "servidor" das aulas).
# Sobe um servidor HTTP nesta própria sessão, em http://127.0.0.1:8765, que
# responde em JSON como a API de um sistema de inventário de rede.
import json
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

PORTA_API = 8765

_EQUIPAMENTOS = [
    {"nome": "OLT-CENTRO-01", "tipo": "OLT", "ip": "10.0.1.10", "localidade": "Centro", "em_servico": True},
    {"nome": "OLT-NORTE-02", "tipo": "OLT", "ip": "10.0.2.10", "localidade": "Zona Norte", "em_servico": True},
    {"nome": "OLT-SUL-03", "tipo": "OLT", "ip": "10.0.3.10", "localidade": "Zona Sul", "em_servico": False},
    {"nome": "SWITCH-CENTRO-01", "tipo": "SWITCH", "ip": "10.0.1.20", "localidade": "Centro", "em_servico": True},
    {"nome": "SWITCH-NORTE-02", "tipo": "SWITCH", "ip": "10.0.2.20", "localidade": "Zona Norte", "em_servico": True},
    {"nome": "RADIO-OESTE-01", "tipo": "RADIO", "ip": "10.0.5.10", "localidade": "Zona Oeste", "em_servico": True},
]
_ALARMES = [
    {"id": 101, "equipamento": "OLT-CENTRO-01", "severidade": "CRITICAL", "mensagem": "perda de sinal na porta GPON0/1/3"},
    {"id": 102, "equipamento": "SWITCH-NORTE-02", "severidade": "ERROR", "mensagem": "Interface Gi0/12, changed state to down"},
    {"id": 103, "equipamento": "RADIO-OESTE-01", "severidade": "WARNING", "mensagem": "enlace degradado"},
    {"id": 104, "equipamento": "OLT-CENTRO-01", "severidade": "WARNING", "mensagem": "temperatura acima do limite"},
    {"id": 105, "equipamento": "RADIO-OESTE-01", "severidade": "CRITICAL", "mensagem": "enlace fora do ar"},
]
_chamadas_instavel = [0]


class _Tratador(BaseHTTPRequestHandler):
    def log_message(self, *args):          # sem log na tela
        pass

    def _responde(self, status, corpo):
        dados = json.dumps(corpo, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()
        self.wfile.write(dados)

    def do_GET(self):
        url = urlparse(self.path)
        partes = [p for p in url.path.split("/") if p]
        filtros = {chave: valores[0] for chave, valores in parse_qs(url.query).items()}
        if partes == ["api", "saude"]:
            return self._responde(200, {"status": "ok"})
        if partes == ["api", "equipamentos"]:
            itens = [e for e in _EQUIPAMENTOS
                     if all(str(e.get(k)) == v for k, v in filtros.items())]
            return self._responde(200, {"count": len(itens), "results": itens})
        if len(partes) == 3 and partes[:2] == ["api", "equipamentos"]:
            for e in _EQUIPAMENTOS:
                if e["nome"] == partes[2]:
                    return self._responde(200, e)
            return self._responde(404, {"erro": f"equipamento {partes[2]} não encontrado"})
        if partes == ["api", "alarmes"]:
            itens = [a for a in _ALARMES
                     if all(str(a.get(k)) == v for k, v in filtros.items())]
            return self._responde(200, {"count": len(itens), "results": itens})
        if partes == ["api", "lento"]:
            time.sleep(3)
            return self._responde(200, {"status": "finalmente"})
        if partes == ["api", "instavel"]:
            _chamadas_instavel[0] += 1
            if _chamadas_instavel[0] % 2 == 1:
                return self._responde(503, {"erro": "serviço temporariamente indisponível"})
            return self._responde(200, {"status": "ok"})
        return self._responde(404, {"erro": f"rota {url.path} não existe"})


def inicia_api(porta=PORTA_API):
    """Sobe a API simulada (se ainda não estiver no ar) e devolve o endereço."""
    endereco = f"http://127.0.0.1:{porta}"
    try:
        urllib.request.urlopen(endereco + "/api/saude", timeout=1)
        return endereco                     # já estava no ar
    except OSError:
        pass
    servidor = ThreadingHTTPServer(("127.0.0.1", porta), _Tratador)
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    return endereco


API = inicia_api()
print("API simulada no ar em", API)

## 📝 Mini-teste — dez minutos

Caderno fechado. O **Mini-teste 13**, no papel, cobre as duas noites da Aula 12:
rastrear um trecho curto e escrever uma função pequena. Depois dele, o chamado de
hoje.

> 💡 **Pense assim: o garçom.**
>
> No restaurante, você não entra na cozinha: faz o pedido ao garçom, num formato que
> ele entende ("um prato do dia, sem cebola"), e ele volta com o prato ou com um
> aviso ("acabou"). Uma **API** é o garçom de um sistema: o seu programa faz o
> pedido pela rede, e recebe a resposta em JSON. O **servidor** é a cozinha — o
> programa que fica esperando pedidos e respondendo. Veja
> [Conversar com os equipamentos](https://lacouth.github.io/python_telecom-site/unidade0-primeiros-passos/rede-marenet/#conversar-com-os-equipamentos-servidor-api-e-ssh).

> 📡 **Na rede: inventário na rede.**
>
> O **NetBox** é um sistema de inventário: a lista oficial de todos os equipamentos
> de um provedor, com os dados de cada um, guardada num servidor e consultada por
> toda a equipe e pelos scripts. É o "cadastro central" da rede. A API simulada da
> Maré Net responde no mesmo formato que ele.

## 📟 O chamado de hoje

> **Chamado #1502 — NOC Maré Net**
>
> *"Estagiário, o sistema de inventário agora tem uma API. Quero um **painel
> simples**: para cada equipamento **em serviço**, quantos alarmes **críticos** ele
> tem abertos. Hoje alguém entra na tela do sistema e conta um por um."*

Na noite B o script faz isso perguntando direto ao sistema.

## 1. Requisição e resposta

A **requisição** leva o método (`GET`, para ler), a URL e, se houver, parâmetros. A
**resposta** traz o código de status (`200` = deu certo), cabeçalhos e o corpo —
quase sempre JSON. Em Python, quem faz a requisição é a biblioteca `requests`.

📖 [capítulo 13 · Requisição e resposta](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#requisicao-e-resposta)

> 💡 **Pense assim: o endereço completo.**
>
> Uma **URL** é um endereço completo, lido da esquerda para a direita como um
> endereço postal: `http://127.0.0.1:8765/api/equipamentos/OLT-CENTRO-01` é o jeito
> de falar (`http`), o prédio (`127.0.0.1`, o endereço IP do servidor), o número da
> sala (`8765`, a **porta** — um mesmo computador atende vários serviços, cada um na
> sua porta) e, por fim, o caminho até o recurso, como andar, corredor e armário.
> `127.0.0.1` é um endereço especial: quer dizer "este próprio computador" — é onde o
> simulador está rodando.

> 💡 **Pense assim: desistir da fila.**
>
> O `timeout` é decidir, ao entrar na fila do banco, "se em 5 minutos eu não for
> atendido, vou embora e volto outra hora". Sem essa decisão, você fica na fila
> para sempre se o caixa tiver saído para almoçar — e é isso que um script sem
> `timeout` faz quando o servidor não responde.

**✍️ Passo 1.** Escreva `import requests` e faça
`resposta = requests.get(API + "/api/equipamentos/OLT-CENTRO-01", timeout=5)`. Imprima
`resposta.status_code` e `resposta.headers["Content-Type"]`.

In [ ]:
# ✍️ passo 1

**Preveja:** que código de status sai? E o que o `Content-Type` diz?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`200` — deu certo — e `application/json; charset=utf-8`: o corpo é JSON. O `timeout=5`
diz para desistir depois de 5 segundos; **sempre** passe um, ou uma API que não
responde deixa o script parado para sempre.

</details>

**✍️ Passo 2.** Faça `equipamento = resposta.json()` e imprima `type(equipamento)` e
`equipamento["ip"]`.

In [ ]:
# ✍️ passo 2

**Preveja:** o que o `.json()` devolve?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Um **dicionário** (`<class 'dict'>`) e o IP `10.0.1.10`. O `.json()` é o `json.loads`
da Aula 08 aplicado ao corpo da resposta — daí em diante, é o dicionário de sempre.

</details>

### 🎯 Sua vez — O tipo de um equipamento

Escreva `tipo_de(api, nome)`, que consulta `/api/equipamentos/<nome>` e devolve o campo
`tipo` — ou `None` se o código de status não for `200`.

In [ ]:
def tipo_de(api, nome):
    # sua solução aqui
    pass

In [ ]:
confere(tipo_de, [
    ((API, "OLT-CENTRO-01"), "OLT"),
    ((API, "RADIO-OESTE-01"), "RADIO"),
    ((API, "OLT-XYZ"), None),
])

<details>
<summary><b>💡 Dica</b></summary>

`requests.get(api + "/api/equipamentos/" + nome, timeout=5)`; **se** o `status_code`
for diferente de `200`, devolva `None`; senão, `resposta.json()["tipo"]`.

</details>

## 2. Uma lista de recursos

Sem o nome no fim, `/api/equipamentos` devolve **todos**, no formato do NetBox: a
quantidade em `count` e os itens em `results` — uma lista de dicionários.

📖 [capítulo 13 · Uma lista de recursos](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#uma-lista-de-recursos)

**✍️ Passo 3.** Faça `dados = requests.get(API + "/api/equipamentos", timeout=5).json()`. Imprima
`dados["count"]` e, num laço sobre `dados["results"]`, o nome e o IP de cada um.

In [ ]:
# ✍️ passo 3

**Preveja:** o que é `dados["results"]`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`6` e seis linhas com nome e IP. `dados["results"]` é uma **lista de dicionários** —
o inventário da Aula 06, chegando pela rede.

</details>

## 3. Filtrar no servidor: parâmetros

`params={"tipo": "OLT"}` vira a parte depois do `?` da URL, e quem filtra é o
**servidor** — ele devolve só o que foi pedido.

📖 [capítulo 13 · Filtrar no servidor: parâmetros](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#filtrar-no-servidor-parametros)

> 💡 **Pense assim: o pedido ao bibliotecário.**
>
> Pedir ao bibliotecário "só os livros de redes" é muito mais rápido do que pedir
> todos os livros da biblioteca e separar em casa. Os `params` são o "só os de
> redes" do pedido: quem filtra é o servidor, que conhece o acervo.

**✍️ Passo 4.** Faça `resposta = requests.get(API + "/api/equipamentos", params={"tipo": "OLT"}, timeout=5)`.
Imprima `resposta.url` e `resposta.json()["count"]`.

In [ ]:
# ✍️ passo 4

**Preveja:** como fica a URL montada?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`http://127.0.0.1:8765/api/equipamentos?tipo=OLT` e `3`. Com seis equipamentos não faz
diferença; com os 40 mil de uma operadora, filtrar no servidor evita trazer tudo para
jogar a maior parte fora.

</details>

## 4. O código de status

Pedir um recurso que não existe **não** dá erro no Python: a resposta chega, com
código `404`. Conferir o código antes de usar o corpo é responsabilidade do script. A
regra de bolso: começou com **2**, deu certo; com **4**, o problema está no pedido; com
**5**, no servidor.

📖 [capítulo 13 · O código de status](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#o-codigo-de-status)

> 💡 **Pense assim: o aviso dos Correios.**
>
> É como o aviso que volta de uma encomenda: "entregue" (2xx); "endereço não
> existe" ou "destinatário recusou" — o problema está no que você mandou (4xx);
> "agência fechada por falha no sistema" — o problema está do lado de lá (5xx). Cada
> aviso pede uma reação diferente de quem mandou.

**✍️ Passo 5.** Faça `resposta = requests.get(API + "/api/equipamentos/OLT-XYZ", timeout=5)` e imprima
`resposta.status_code`, `resposta.ok` e `resposta.json()`.

In [ ]:
# ✍️ passo 5

**Preveja:** o programa quebra ao pedir um equipamento que não existe?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não quebra: sai `404`, `False` e o corpo `{'erro': 'equipamento OLT-XYZ não
encontrado'}`. Se o script tentasse `resposta.json()["ip"]` sem conferir o código,
aí sim quebraria — com um `KeyError` que não explica nada.

</details>

**✍️ Passo 6.** Agora chame `resposta.raise_for_status()` nessa mesma resposta.

In [ ]:
# ✍️ passo 6

**Preveja:** o que o `raise_for_status` faz com um 404?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`HTTPError: 404 Client Error: Not Found for url: ...`. O `raise_for_status()`
transforma qualquer código 4xx ou 5xx numa **exceção** — útil quando o erro deve
interromper a operação, ou ser capturado por um `try`.

</details>

**✍️ Passo 7.** Peça a rota lenta com um `timeout` curto:
`requests.get(API + "/api/lento", timeout=1)`.

In [ ]:
# ✍️ passo 7

**Preveja:** a rota demora 3 segundos. O que acontece com o timeout de 1?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`ReadTimeout`: a `requests` desiste depois de 1 segundo. Sem o `timeout`, o script
ficaria esperando os 3 segundos — ou para sempre, se a API tivesse travado.

</details>

## 5. Falha por item: o coletor que não morre

O padrão da Aula 08, com a rede no meio: cada consulta dentro do seu `try`.
`requests.RequestException` captura todos os erros da biblioteca — o `HTTPError`, o
`Timeout`, a conexão recusada.

📖 [capítulo 13 · Falha por item: o coletor que não morre](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#falha-por-item-o-coletor-que-nao-morre)

> 💡 **Pense assim: a linha ocupada.**
>
> Deu ocupado? Você liga de novo daqui a pouco. Deu ocupado três vezes? Você desiste
> e manda uma mensagem. Tentar de novo é razoável para uma falha passageira; tentar
> para sempre é ficar a noite inteira ao telefone.

In [ ]:
# 📦 dados prontos — só rode esta célula
nomes = ["OLT-CENTRO-01", "OLT-XYZ", "SWITCH-NORTE-02"]

**✍️ Passo 8.** Crie `ips = {}` e `falhas = []`. Para cada `nome` em `nomes`, dentro de um `try:`: o
`GET` em `/api/equipamentos/` + nome (com `timeout=2`), o `raise_for_status()` e
`ips[nome] = resposta.json()["ip"]`. No `except requests.RequestException:`,
`falhas.append(nome)`. Imprima `ips` e `falhas`.

In [ ]:
# ✍️ passo 8

**Preveja:** o `OLT-XYZ` impede a consulta do `SWITCH-NORTE-02`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: `ips` tem os dois que existem e `falhas` tem `['OLT-XYZ']`. O
`raise_for_status` transformou o 404 em exceção, o `except` registrou e o laço seguiu.

</details>

**✍️ Passo 9.** Um serviço instável: num laço `for tentativa in range(1, 4):`, faça o `GET` em
`API + "/api/instavel"`, imprima `tentativa` e o código, e dê `break` se o código for
`200`.

In [ ]:
# ✍️ passo 9

**Preveja:** por que o `range(1, 4)` — e não um laço que tenta até dar certo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A rota alterna entre `503` e `200`, então em uma ou duas tentativas dá certo. O
`range(1, 4)` é o **limite**: um laço de tentativas sem limite, no dia em que o
serviço cair de verdade, fica preso para sempre.

</details>

> ⚠️ **Armadilha.** Tentar de novo em **qualquer** erro. Um `503` (serviço indisponível) pode passar; um
> `404` (não existe) ou um `401` (sem permissão) não vai mudar na próxima tentativa —
> repetir só gasta tempo e carrega o servidor. Tente de novo só em erro 5xx.

## 6. O que mudou desde ontem

Com o inventário vindo da API, a comparação com a coleta anterior é a dos conjuntos da
Aula 06.

📖 [capítulo 13 · O que mudou desde ontem](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/#o-que-mudou-desde-ontem)

In [ ]:
# 📦 dados prontos — só rode esta célula
ontem = {"OLT-CENTRO-01", "OLT-NORTE-02", "OLT-SUL-03", "SWITCH-CENTRO-01", "ONU-SUL-4512"}

**✍️ Passo 10.** Monte `hoje = set()` com o nome de cada equipamento de `/api/equipamentos`. Imprima
`sorted(hoje - ontem)` e `sorted(ontem - hoje)`.

In [ ]:
# ✍️ passo 10

**Preveja:** quem apareceu e quem sumiu?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Apareceram `RADIO-OESTE-01` e `SWITCH-NORTE-02`; sumiu `ONU-SUL-4512`. Rodar isso todo
dia e mandar a diferença por e-mail é um dos primeiros scripts de automação de uma
equipe de rede — feito inteiro com coisas deste curso.

</details>

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** `requests.get(url)` para um recurso que não existe:
a) levanta erro na hora  b) devolve uma resposta com código 404  c) devolve `None`
d) trava

> 🌉 **Esta fica sem resposta aqui.** Pense nela até o encontro seguinte: é a
> primeira coisa da noite B.

**2.** Por que passar sempre `timeout=` no `requests.get`?
a) deixa mais rápido  b) sem ele, uma API que não responde deixa o script parado para
sempre  c) é obrigatório  d) para o JSON vir formatado

<details>
<summary><b>Resposta da 2</b></summary>

**b**.

</details>

**3.** Um código `503` quer dizer:
a) o pedido está errado  b) o recurso não existe  c) o servidor está indisponível no
momento  d) deu certo

<details>
<summary><b>Resposta da 3</b></summary>

**c** — começou com 5, o problema é do servidor; e pode valer tentar de novo, com
limite.

</details>

## 🏠 Para casa

- **No encontro seguinte — noite B:** laboratório, sem assunto novo. Os 🎯 que
  ficaram, o chamado resolvido e a [Lista
  13](https://lacouth.github.io/python_telecom-site/listas/lista13/) começada em sala.
- Releia no [capítulo 13 do
  site](https://lacouth.github.io/python_telecom-site/unidade7-redes/13-apis-rest/) as
  seções que ficaram difíceis: o link 📖 de cada bloco leva direto a elas.
- Guarde a pergunta 1 do 🚪: ela abre a noite B.